In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

os.getenv("GEMINI_API_KEY")[:8]

'AIzaSyCs'

In [3]:
from google import genai
client = genai.Client()
print("Gemini OK")

Gemini OK


In [4]:
stores = list(client.file_search_stores.list())

print("Total stores:", len(stores))

for s in stores:
    print(s.name, "|", getattr(s, "display_name", None))

Total stores: 1
fileSearchStores/kcwkbstore-y3a6az4kie04 | kcw-kb-store


In [7]:
store = stores[0]

print("Using store:", store.name)

Using store: fileSearchStores/kcwkbstore-y3a6az4kie04


In [9]:
import re
from pathlib import Path

def clean_filename(path: str) -> str:
    name = Path(path).stem

    # remove non-ascii
    name = name.encode("ascii", "ignore").decode()

    # replace bad chars with _
    name = re.sub(r"[^A-Za-z0-9._-]+", "_", name)

    # collapse multiple _
    name = re.sub(r"_+", "_", name)

    # trim
    name = name.strip("_")

    # keep short
    return name[:60] or "doc"

In [11]:
from pathlib import Path
import time
import json
import os
import requests

KB_FOLDER = Path(r"G:\Shared drives\KCW-Data\kcw_analytics\03_curated\knowledge_based")
REGISTRY_FILE = KB_FOLDER / "_uploaded_registry.json"

API_KEY = os.environ["GEMINI_API_KEY"]  # make sure this is set
STORE_NAME = store.name                  # e.g. "fileSearchStores/xxxx"

BASE_URL = "https://generativelanguage.googleapis.com/v1beta"
HEADERS = {"x-goog-api-key": API_KEY}


def list_store_documents(store_name: str) -> list[dict]:
    """Return all documents currently in the file search store."""
    docs = []
    page_token = None

    while True:
        params = {"pageSize": 20}
        if page_token:
            params["pageToken"] = page_token

        url = f"{BASE_URL}/{store_name}/documents"
        resp = requests.get(url, headers=HEADERS, params=params, timeout=60)
        resp.raise_for_status()

        data = resp.json()
        docs.extend(data.get("documents", []))

        page_token = data.get("nextPageToken")
        if not page_token:
            break

    return docs


def delete_store_document(doc_name: str) -> None:
    """Delete a document from the file search store."""
    url = f"{BASE_URL}/{doc_name}"
    resp = requests.delete(url, headers=HEADERS, params={"force": "true"}, timeout=60)
    resp.raise_for_status()


def wait_operation(op):
    """Poll long-running upload/import operation until done."""
    while not op.done:
        time.sleep(5)
        op = client.operations.get(op)

    # optional: raise if the operation completed with an error
    if getattr(op, "error", None):
        raise RuntimeError(f"Operation failed: {op.error}")

    return op


# ---------- local files ----------
local_files = {
    p.name: p
    for p in KB_FOLDER.glob("*.md")
}

print(f"Local .md files: {len(local_files)}")

# ---------- remote docs ----------
remote_docs = list_store_documents(STORE_NAME)

# map by displayName, since you uploaded with config={"display_name": name}
remote_by_display_name = {}
for d in remote_docs:
    display_name = d.get("displayName") or ""
    if display_name:
        remote_by_display_name[display_name] = d

print(f"Remote documents: {len(remote_docs)}")
print(f"Remote documents with displayName: {len(remote_by_display_name)}")

local_names = set(local_files.keys())
remote_names = set(remote_by_display_name.keys())

to_upload = sorted(local_names - remote_names)
to_delete = sorted(remote_names - local_names)
to_keep   = sorted(local_names & remote_names)

print("\n--- SYNC PLAN ---")
print("Keep   :", len(to_keep))
print("Upload :", len(to_upload))
print("Delete :", len(to_delete))

# ---------- delete docs that no longer exist locally ----------
for name in to_delete:
    doc = remote_by_display_name[name]
    doc_name = doc["name"]   # e.g. fileSearchStores/.../documents/...

    print("Deleting:", name)
    delete_store_document(doc_name)
    print("Deleted :", name)

# ---------- upload new local files ----------
for name in to_upload:
    file_path = local_files[name]

    print("Uploading:", name)

    op = client.file_search_stores.upload_to_file_search_store(
        file=str(file_path),
        file_search_store_name=STORE_NAME,
        config={
            "display_name": name,
            "mime_type": "text/markdown"
        }
    )

    wait_operation(op)
    print("Indexed  :", name)

# ---------- rewrite registry to exactly match local folder ----------
REGISTRY_FILE.write_text(
    json.dumps(sorted(local_names), indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n✅ SYNC COMPLETE")
print(f"Registry updated: {REGISTRY_FILE}")

Local .md files: 28
Remote documents: 28
Remote documents with displayName: 28

--- SYNC PLAN ---
Keep   : 27
Upload : 1
Delete : 1
Deleting: parts_knowledge_CrossSell_product set_29032026.md
Deleted : parts_knowledge_CrossSell_product set_29032026.md
Uploading: parts_knowledge_CrossSell_set_29032026.md
Indexed  : parts_knowledge_CrossSell_set_29032026.md

✅ SYNC COMPLETE
Registry updated: G:\Shared drives\KCW-Data\kcw_analytics\03_curated\knowledge_based\_uploaded_registry.json


In [19]:
# FULL_SYSTEM_PROMPT = """
# บทบาท:
# คุณคือผู้เชี่ยวชาญด้านเทคนิคอะไหล่ยนต์และเครื่องจักร ตอบโดยอ้างอิงข้อมูลจากเอกสารในคลัง File Search เท่านั้น

# รูปแบบการตอบ (LINE):
# - ตอบภาษาไทย สั้น กระชับ อ่านง่าย
# - บรรทัดแรก: สรุปคำตอบตรงๆ ไม่เกิน 2 ประโยค
# - จากนั้นสรุปเป็นหัวข้อสั้นๆ 2–4 ข้อ
# - หากมีข้อควรระวัง ให้ใส่ท้ายคำตอบแบบสั้น

# กฎสำคัญ:
# - ยึดข้อมูลจากเอกสารเป็นหลัก สามารถเชื่อมโยงคำที่มีความหมายใกล้เคียงได้ แต่ห้ามสร้างข้อมูลทางเทคนิคใหม่
# - หากข้อมูลไม่ชัดเจน ให้ตอบว่า "ข้อมูลในคลังไม่ชัดเจน"
# - หากไม่พบข้อมูล ให้ตอบว่า "ไม่มีข้อมูลในคลังข้อมูล"

# การแสดงรูปภาพ:
# - หากพบ Markdown รูปภาพในเอกสาร (เช่น ![alt](url)) ต้องคัดลอกมาแสดงแบบเดิมทันที ห้ามแก้ไข ห้ามย่อ ห้ามละเว้น

# คำถามทั่วไป:
# - หากไม่เกี่ยวกับงานหรืออะไหล่ ให้ตอบเชิงติดตลกว่าเป็นเวลางาน ให้ตั้งใจทำงานก่อน
# """

In [21]:
# from google.genai import types

# response = client.models.generate_content(
#     model="gemini-2.5-flash",
#     contents="คำถามผู้ใช้: ซีลไทตัน",
#     config=types.GenerateContentConfig(
#         system_instruction=FULL_SYSTEM_PROMPT,
#         tools=[
#             types.Tool(
#                 file_search=types.FileSearch(
#                     file_search_store_names=[store.name]
#                 )
#             )
#         ]
#     )
# )

# print(response.text)
# print(response.usage_metadata)